# MLOps on GCP — Hands-On Lab
### GCP Training Program — Day 5, Module 7: MLOps on GCP

**What this notebook covers:** a small live Vertex AI Pipeline, Model Registry (versioning), a
GenAI evaluation run, and guided walkthroughs of CI/CD for ML and Model Monitoring.

## ⚠️ Cost & trial-account notes
The pipeline in this notebook reuses the same tiny Iris dataset and training logic from the Day 4
Vertex AI Platform Core lab, deliberately kept small so the whole pipeline runs in a few minutes.
Model Monitoring is **not run live** — meaningful drift/skew signals need sustained real traffic
over time, which a short class session can't produce; that section is a guided walkthrough with
the exact configuration code, using a pre-existing monitoring dashboard as the visual reference.

**This notebook is fully self-contained.** Authenticate → Configure → everything else runs
against your own project.


## 1. Authenticate This Session

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures gcloud for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth application-default login")

## 2. Install & Import
`kfp` (Kubeflow Pipelines SDK) builds the pipeline definition; `google-cloud-pipeline-components`
provides ready-made Vertex AI pipeline steps; `google-cloud-aiplatform` covers Model Registry and
pipeline execution.

In [ ]:
!pip install --quiet "kfp>=2.7.0" "google-cloud-pipeline-components>=2.14.0" "google-cloud-aiplatform>=1.60.0" scikit-learn

In [ ]:
import time
import os
from google.cloud import aiplatform

print("Imports successful")

## 3. Configure Your Project & Region

In [ ]:
PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your Vertex AI region [default: us-central1]: ").strip()
if not REGION:
    REGION = "us-central1"

_suffix = int(time.time())
STAGING_BUCKET = f"gs://{PROJECT_ID}-mlops-staging-{_suffix}"

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

!gcloud config set project {PROJECT_ID}
!gsutil mb -l {REGION} {STAGING_BUCKET}

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)
print("Vertex AI SDK initialized. Staging bucket:", STAGING_BUCKET)

### Known Issue: Vertex AI Pipelines Permission Error
**If Section 4.3 later fails with:**
```
Permission 'aiplatform.metadataStores.get' denied on resource
'.../metadataStores/default' (or it may not exist)
```
**this is why:** Vertex AI Pipelines needs to read a Metadata Store to track lineage, which
requires `roles/aiplatform.user` on whichever account is actually calling the API. In Colab, that
account is **not necessarily** the one `gcloud auth list` shows as active — `auth.authenticate_user()`
sets up separate credentials (Application Default Credentials, ADC) for Python client libraries
like `google-cloud-aiplatform`, which can be a different identity than the `gcloud` CLI's own
credential store. Granting the role to the wrong one looks like it worked but doesn't fix anything.

**The cell below fixes this properly**, in two steps:
1. Asks Google directly which identity ADC is actually using, via the OAuth tokeninfo endpoint —
   not `gcloud auth list`, and not `credentials.service_account_email`, since Colab's user
   credentials can carry a service-account-*like* attribute even for a real personal login,
   which misclassifies a genuine user as a service account if you trust that attribute directly.
2. Classifies the identity as a user or service account by checking whether the email itself ends
   in `.gserviceaccount.com` — a reliable signal regardless of credential object quirks — then
   grants `roles/aiplatform.admin` to the correctly-typed principal, and verifies the grant actually
   succeeded before claiming it did.

Run it once per new project/session, before Section 4.3.

**Why `aiplatform.admin` and not just `aiplatform.user`:** on a brand-new project, the first pipeline run needs to *create* the default Metadata Store, not just read it — per Google's own docs, this specifically needs `aiplatform.metadataStores.create`, which is not reliably covered by `roles/aiplatform.user` alone in every project configuration. `aiplatform.admin` is broader than strictly necessary, but a reasonable one-time step to unblock a training environment.

**A third identity is involved too — the execution identity.** Submitting a pipeline job and
*running* it are done by two different principals: the account above submits the job, but each
pipeline step actually executes as the **Compute Engine default service account**
(`PROJECT_NUMBER-compute@developer.gserviceaccount.com`), unless a different one is explicitly
passed. If that execution identity can't read/write the staging bucket or the metadata store,
you get the same `metadataStores.get` error even when the submitting identity is completely
correct. The cell below also grants this account `roles/aiplatform.user` and
`roles/storage.objectAdmin`, and Section 4.3 now passes it explicitly via `service_account=...`
instead of relying on the implicit default.

In [ ]:
import google.auth
import google.auth.transport.requests
import urllib.request
import json

credentials, adc_project = google.auth.default()
credentials.refresh(google.auth.transport.requests.Request())

# Ask Google directly which identity this token belongs to — works uniformly for both user and
# service account credentials. We do NOT rely on credentials.service_account_email to tell them
# apart: Colab's authenticated user credentials can carry a service_account_email-like attribute
# even for a genuine personal login, which misclassifies a real user as a service account.
tokeninfo_url = f"https://oauth2.googleapis.com/tokeninfo?access_token={credentials.token}"
with urllib.request.urlopen(tokeninfo_url) as resp:
    token_info = json.load(resp)
calling_identity = token_info.get("email")

# Classify by the email's own domain suffix instead — the one reliable signal IAM itself uses
# to distinguish a user principal ("user:...") from a service account principal ("serviceAccount:...").
if calling_identity and calling_identity.endswith(".gserviceaccount.com"):
    member_type = "serviceAccount"
else:
    member_type = "user"

print("ADC project:            ", adc_project)
print("Actual calling identity:", calling_identity, f"({member_type})")

if calling_identity:
    grant_result = !gcloud projects add-iam-policy-binding {PROJECT_ID} \
        --member="{member_type}:{calling_identity}" \
        --role="roles/aiplatform.admin" \
        --condition=None \
        --quiet 2>&1
    grant_output = "\n".join(grant_result)

    # Check whether it actually succeeded — the earlier version of this cell printed a success
    # message unconditionally, which was wrong: it claimed success even when gcloud had just
    # printed an ERROR line above it.
    if "ERROR" in grant_output.upper():
        print("\nGrant FAILED — gcloud reported:")
        print(grant_output)
        print("\nIf this still shows a type mismatch, double-check member_type against the")
        print("identity printed above, or use the Troubleshooter link from the Section 4.3 error.")
    else:
        print(f"\nGranted roles/aiplatform.admin to {member_type}:{calling_identity}")
        print("If Section 4.3 was already failing, allow 1-2 minutes for this to propagate before retrying.")
else:
    print("Could not determine the calling identity automatically — if Section 4.3 fails with a")
    print("metadataStores permission error, use the Troubleshooter link in that error message,")
    print("which will name the exact identity and missing permission for your project.")

# --- Also cover the pipeline's EXECUTION identity, not just the submission identity ---
# pipeline_job.run() submits as the identity above, but Vertex AI actually EXECUTES each
# pipeline step as a separate principal: the Compute Engine default service account, unless
# a different one is explicitly passed via service_account=... in Section 4.3. If that
# execution identity can't read/write the staging bucket or the metadata store, pipeline
# creation can fail with the same metadataStores.get error even when the submitting
# identity above is fully correct.
project_number_result = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
project_number = project_number_result[0].strip() if project_number_result else ""

if project_number:
    compute_sa = f"{project_number}-compute@developer.gserviceaccount.com"
    for role in ["roles/aiplatform.user", "roles/storage.objectAdmin"]:
        compute_grant_result = !gcloud projects add-iam-policy-binding {PROJECT_ID} \
            --member="serviceAccount:{compute_sa}" \
            --role="{role}" \
            --condition=None \
            --quiet 2>&1
        compute_grant_output = "\n".join(compute_grant_result)
        if "ERROR" in compute_grant_output.upper():
            print(f"\nGrant of {role} to the compute default SA FAILED:")
            print(compute_grant_output)
        else:
            print(f"Granted {role} to {compute_sa}")
else:
    print("Could not determine the project number — grant roles/aiplatform.user and")
    print("roles/storage.objectAdmin to your PROJECT_NUMBER-compute@developer.gserviceaccount.com")
    print("manually if Section 4.3 keeps failing after the grant above.")

## 4. Vertex AI Pipelines
### 4.1 Define a 3-Step Pipeline as Lightweight Python Components
**What this does:** each `@component` decorator turns a plain Python function into a containerized
pipeline step. Using `base_image="python:3.10"` with inline `pip install` keeps these lightweight —
no custom Docker image to build, which is what makes this feasible to run live in a few minutes
rather than the 20-30+ minutes a custom-container pipeline typically takes to first build.

**Step 1 — generate data:** creates the tiny Iris dataset and writes it as a pipeline artifact.
**Step 2 — train:** trains a small RandomForest model against that artifact.
**Step 3 — evaluate:** scores the trained model and outputs an accuracy metric.

In [ ]:
from kfp import dsl
from kfp.dsl import component, Output, Input, Dataset, Model, Metrics


@component(base_image="python:3.10", packages_to_install=["scikit-learn", "pandas"])
def generate_data(output_dataset: Output[Dataset]):
    from sklearn.datasets import load_iris
    import pandas as pd

    X, y = load_iris(return_X_y=True, as_frame=True)
    df = X.copy()
    df["target"] = y
    df.to_csv(output_dataset.path, index=False)
    print(f"Generated {len(df)} rows of Iris data.")


@component(base_image="python:3.10", packages_to_install=["scikit-learn", "pandas", "joblib"])
def train_model(input_dataset: Input[Dataset], output_model: Output[Model]):
    import pandas as pd
    import joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split

    df = pd.read_csv(input_dataset.path)
    X = df.drop(columns=["target"])
    y = df["target"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
    model.fit(X_train, y_train)

    output_model.metadata["test_size"] = len(X_test)
    joblib.dump(model, output_model.path + ".joblib")
    print("Model trained.")


@component(base_image="python:3.10", packages_to_install=["scikit-learn", "pandas", "joblib"])
def evaluate_model(
    input_dataset: Input[Dataset],
    input_model: Input[Model],
    output_metrics: Output[Metrics],
):
    import pandas as pd
    import joblib
    from sklearn.model_selection import train_test_split

    df = pd.read_csv(input_dataset.path)
    X = df.drop(columns=["target"])
    y = df["target"]
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = joblib.load(input_model.path + ".joblib")
    accuracy = model.score(X_test, y_test)

    output_metrics.log_metric("accuracy", accuracy)
    print(f"Evaluation accuracy: {accuracy:.4f}")


print("Defined 3 lightweight pipeline components: generate_data -> train_model -> evaluate_model")

### 4.2 Assemble & Compile the Pipeline
**What this does:** `@dsl.pipeline` wires the three components together — note how
`train_model`'s input is `data_step.outputs["output_dataset"]`, i.e. pipeline steps pass data
between each other as tracked artifacts, not just return values. `compile()` turns this Python
definition into a JSON pipeline spec that Vertex AI Pipelines can actually execute.

In [ ]:
from kfp import compiler


@dsl.pipeline(name="iris-training-pipeline", description="Tiny 3-step training pipeline for the MLOps lab")
def iris_pipeline():
    data_step = generate_data()
    train_step = train_model(input_dataset=data_step.outputs["output_dataset"])
    evaluate_model(
        input_dataset=data_step.outputs["output_dataset"],
        input_model=train_step.outputs["output_model"],
    )


compiler.Compiler().compile(pipeline_func=iris_pipeline, package_path="iris_pipeline.json")
print("Compiled pipeline to iris_pipeline.json")

### 4.3 Submit the Pipeline Run
**Cost/time note:** each of the 3 steps runs as its own small managed job — expect roughly
5-8 minutes total including container startup overhead for each step, even though the actual work
in each step takes only seconds. `sync=True` blocks the cell so the class can watch it progress.

In [ ]:
pipeline_job = aiplatform.PipelineJob(
    display_name=f"iris-pipeline-run-{_suffix}",
    template_path="iris_pipeline.json",
    pipeline_root=STAGING_BUCKET,
)

# Explicitly pass the execution service account rather than relying on the implicit default —
# this is what Section 3's fix cell grants roles to, and being explicit here means the pipeline
# never silently falls back to a different, ungranted identity if the project's default changes.
pipeline_job.run(sync=True, service_account=compute_sa)
print("Pipeline run complete. View the DAG in the console under Vertex AI > Pipelines.")

## 5. Model Registry
### 5.1 Register the Trained Model
**What this does:** takes the model artifact the pipeline just produced and registers it in Vertex
AI Model Registry — the versioned, governed home for models that Prediction/Endpoints and
CI/CD tooling reference by name rather than by a raw file path.

In [ ]:
MODEL_DISPLAY_NAME = f"iris-model-{_suffix}"

model_artifact_uri = f"{STAGING_BUCKET}/{pipeline_job.gca_resource.job_id}"

# In a real pipeline you would export the model to a fixed GCS path from within train_model.
# For this lab, we re-run the tiny training step directly and upload its output for registration —
# keeping the registry step decoupled and easy to re-run independently of the pipeline above.
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib

X, y = load_iris(return_X_y=True)
quick_model = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42).fit(X, y)
joblib.dump(quick_model, "model.joblib")

MODEL_GCS_DIR = f"{STAGING_BUCKET}/registered-model-v1"
!gsutil cp model.joblib {MODEL_GCS_DIR}/model.joblib

registered_model = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    artifact_uri=MODEL_GCS_DIR,
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest",
)
print("Registered model:", registered_model.resource_name)
print("Version ID:", registered_model.version_id)

### 5.2 Register a Second Version & List Version History
**What this does:** uploading again with `parent_model=` creates a **new version of the same
model** rather than an unrelated new model — this is how Model Registry supports comparing,
rolling back to, or gradually rolling out between versions.

In [ ]:
MODEL_GCS_DIR_V2 = f"{STAGING_BUCKET}/registered-model-v2"
!gsutil cp model.joblib {MODEL_GCS_DIR_V2}/model.joblib

registered_model_v2 = aiplatform.Model.upload(
    display_name=MODEL_DISPLAY_NAME,
    artifact_uri=MODEL_GCS_DIR_V2,
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest",
    parent_model=registered_model.resource_name,
    is_default_version=True,
)

print("Registered version:", registered_model_v2.version_id, "(now the default version)")

all_versions = aiplatform.Model.list(filter=f'display_name="{MODEL_DISPLAY_NAME}"')
print(f"\n{len(all_versions)} model(s) sharing this display name (Model Registry groups versions under one entry).")

## 6. CI/CD for ML (Guided Walkthrough)
**Why this is a walkthrough, not a live exercise:** a full CI/CD demo needs a connected Git repo
and a configured Cloud Build trigger — real infrastructure setup that eats class time without
teaching anything beyond "click these console buttons." Instead, here's the actual
`cloudbuild.yaml` such a trigger would run — walk through what each step does.

In [ ]:
cloudbuild_yaml = """\
steps:
  # 1. Compile the KFP pipeline definition (same compiler.Compiler() call as Section 4.2)
  - name: "python:3.10"
    entrypoint: bash
    args:
      - -c
      - |
        pip install --quiet kfp google-cloud-pipeline-components
        python pipelines/compile_pipeline.py

  # 2. Submit the compiled pipeline to Vertex AI Pipelines
  - name: "gcr.io/google.com/cloudsdktool/cloud-sdk"
    entrypoint: bash
    args:
      - -c
      - |
        pip install --quiet google-cloud-aiplatform
        python pipelines/submit_pipeline_job.py --project_id=$PROJECT_ID --region=$_REGION

substitutions:
  _REGION: us-central1

options:
  logging: CLOUD_LOGGING_ONLY
"""

with open("cloudbuild.yaml", "w") as f:
    f.write(cloudbuild_yaml)

print(cloudbuild_yaml)
print("\nIn a real setup: a Cloud Build TRIGGER (console or 'gcloud builds triggers create github')")
print("watches a repo branch and runs this file automatically on every push — retraining and")
print("redeploying the pipeline as code changes, the same principle as any other CI/CD pipeline.")

## 7. Model Monitoring (Guided Walkthrough)
**Why this is a walkthrough, not a live exercise:** meaningful drift/skew detection needs a
deployed endpoint receiving *real, varying* traffic over hours or days — a monitoring job enabled
5 minutes ago on an endpoint with no traffic has nothing to show. This is the exact configuration
code you'd use on a real deployed endpoint; walk through it, then show a monitoring dashboard
screenshot from a model that's had traffic for a while if you have one available.

In [ ]:
# REFERENCE ONLY — assumes `my_endpoint` is a real, already-deployed Vertex AI Endpoint
# receiving live prediction traffic. Not executed here since no traffic exists yet.

# from google.cloud.aiplatform import model_monitoring
#
# skew_config = model_monitoring.SkewDetectionConfig(
#     data_source="gs://your-bucket/training-data.csv",
#     skew_thresholds={"sepal_length": 0.3, "petal_length": 0.3},
#     target_field="target",
# )
#
# drift_config = model_monitoring.DriftDetectionConfig(
#     drift_thresholds={"sepal_length": 0.3, "petal_length": 0.3},
# )
#
# objective_config = model_monitoring.ObjectiveConfig(
#     skew_detection_config=skew_config,
#     drift_detection_config=drift_config,
# )
#
# my_endpoint.create_monitoring_job(
#     objective_configs=objective_config,
#     alert_config=model_monitoring.EmailAlertConfig(user_emails=["you@example.com"]),
#     schedule_config=model_monitoring.ScheduleConfig(monitor_interval=3600),  # hourly checks
# )

print("Reference code above — walk through it conceptually; it needs a live-traffic endpoint to be meaningful.")

## 8. GenAI Evaluation Framework
### 8.1 Build a Tiny Evaluation Dataset
**What this does:** a handful of prompt/reference-answer pairs stand in for a real evaluation set —
the point is the evaluation *mechanism*, not dataset size.

In [ ]:
!pip install --quiet "google-cloud-aiplatform[evaluation]>=1.60.0"

In [ ]:
import pandas as pd
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project=PROJECT_ID, location=REGION)

eval_dataset = pd.DataFrame({
    "prompt": [
        "What does BigQuery partitioning do?",
        "What is the fan-out pattern in Pub/Sub?",
        "What does a materialized view store?",
    ],
    "reference": [
        "It splits a table into segments by date or range so queries can skip irrelevant segments.",
        "Every subscription attached to a topic receives its own independent copy of each message.",
        "A materialized view stores the precomputed result of a query, refreshed incrementally.",
    ],
})
print(eval_dataset)

### 8.2 Generate Model Responses & Run Evaluation Metrics
**What this does:** generates a real Gemini response for each prompt, then scores each response
against its reference answer using the Vertex AI Rapid Evaluation SDK's built-in metrics —
automating what would otherwise be manual side-by-side comparison.

In [ ]:
model = GenerativeModel("gemini-1.5-flash")
eval_dataset["response"] = eval_dataset["prompt"].apply(lambda p: model.generate_content(p).text)

from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

eval_task = EvalTask(
    dataset=eval_dataset,
    metrics=["bleu", "rouge_l_sum", MetricPromptTemplateExamples.Pointwise.COHERENCE],
)

result = eval_task.evaluate()
print("Summary metrics:")
print(result.summary_metrics)

## 9. Cleanup
Deletes the registered model versions, the pipeline's staging artifacts, and the staging bucket.

In [ ]:
registered_model.delete()
print("Deleted registered model (all versions).")

!gsutil -m rm -r {STAGING_BUCKET}
print(f"Deleted staging bucket {STAGING_BUCKET}.")